# Team MLB · SVM 추가 실험 3·4·5

| 실험 | 질문 | 데이터 |
| --- | --- | --- |
| 3. 하이퍼파라미터 적정성 | SVM은 왜 튜닝 전후 차이가 큰가, 현재 설정이 비교에 적절한가 | Closed-world, trim0 |
| 4. 동일 조건 비교 | LR·RF와 **같은 입력**(trim0, 특징 110개)에서 SVM 기준 성능 | Closed / Open 이진 / Open 다중 |
| 5. 이진 임계값 분석 | 탐지율(TPR)과 오탐률(FPR)의 균형 | Open 이진, trim0 |

**공통 조건 (RF·LR과 동일)**
- 입력: 정보 열 6개를 제외한 특징 **110개** (g1 3 / g2 17 / g3 6 / g4 3 / g5 31 / g6 50), trim0, g6 50지점
  - 기존 SVM 실험은 값이 같은 `g5_iat_pos_max`를 빼고 109개를 썼으나, 실험 4는 다른 모델과 입력을 완전히 맞추기 위해 110개를 그대로 쓴다.
- 분할: `split_seed42.csv` (mon 15,200/3,800, unmon 8,000/2,000)
- 교차검증: `StratifiedKFold(5, shuffle=True, random_state=42)`
- 모델 선택: 과제별 평균 CV 정확도가 가장 높은 설정
- SVM 고유 전처리(학습 구간에서만 적합): g1~g5 `log1p` → 상수 특징 제거 → 표준화. 거리 기반 모델이라 필요한 처리이며, RF가 전처리를 하지 않는 것과 같은 성격의 모델 차이로 본다.
- 최종 평가(test) 기록은 설정·임계값 선택에 쓰지 않고 마지막에 한 번만 사용한다.

> **Open-world 해석 주의**: 조교님 공지에 따르면 0~13번 패킷에 unmonitored 축소 과정에서 생긴 수집 흔적이 있다. trim0 특징에는 이 흔적이 섞여 있으므로(g4가 가장 직접적, g6 누적값에도 작게 남음), Open-world 결과는 다른 모델과의 **같은 조건 비교용**으로 쓰고 사이트 식별 능력으로 직접 해석하지 않는다.

## 0. 설정

In [1]:
FEATURE_DIR = "/content/drive/MyDrive/MLB-Project/feature"
RESULT_DIR  = "/content/drive/MyDrive/MLB-Project/results/svm"
SPLIT_PATH  = "/content/drive/MyDrive/MLB-Project/results/split_seed42.csv"   # LR과 같은 분할 파일
UNMON_NAME  = "unmon10k"
TRIM        = 0
TRIMS_TO_COMPARE = [0, 10, 20]

SEED     = 42
N_FOLDS  = 5
N_JOBS   = -1

# C·gamma 탐색: log2 단위. 1차는 2칸 간격, 최적값이 경계면 범위를 넓히고, 마지막에 1칸 간격으로 주변을 세밀 탐색
LOG2_C_COARSE = [1, 3, 5, 7, 9, 11]
LOG2_G_COARSE = [-11, -9, -7, -5, -3, -1]

RUN_ABLATION = True    # 9번 그룹 제거 실험

In [2]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(RESULT_DIR, exist_ok=True)
print("FEATURE_DIR:", FEATURE_DIR)
print("RESULT_DIR :", RESULT_DIR)
print("SPLIT_PATH :", SPLIT_PATH)

FEATURE_DIR: /content/drive/MyDrive/MLB-Project/feature
RESULT_DIR : /content/drive/MyDrive/MLB-Project/results/svm
SPLIT_PATH : /content/drive/MyDrive/MLB-Project/feature/split_seed42.csv


In [3]:
import time, itertools
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 1. CSV 로드 · 점검

- 열은 정보 6개 + 특징 110개여야 한다. NaN·무한대가 있으면 데이터팀에 알린다.
- `g5_iat_pos_max`와 `g5_iat_all_max`는 모든 기록에서 값이 같다(데이터팀 문서 8-4). SVM에서는 같은 정보가 거리 계산에 두 번 들어가므로 `g5_iat_pos_max`를 뺀다. **SVM의 "전체 특징"은 109개**다.
- log 변환 대상: 0 이상이고 분포가 한쪽으로 긴 g1~g5 특징. 시간 간격 사분위수는 왜도가 100을 넘는다. g6 누적값은 음수가 있어 제외한다.

In [4]:
INFO_COLS = ["record_id", "dataset", "page_id", "sample_idx", "label", "trim"]
DUP_DROP  = ["g5_iat_pos_max"]

def load_features(kind, trim):
    path = os.path.join(FEATURE_DIR, f"features_{kind}_trim{trim}.csv")
    df = pd.read_csv(path)
    feat = [c for c in df.columns if c[:3] in ("g1_", "g2_", "g3_", "g4_", "g5_", "g6_")]
    assert list(df.columns[:6]) == INFO_COLS, f"정보 열 순서가 다름: {path}"
    assert len(feat) == 110, f"특징 수 {len(feat)}개: {path}"
    X = df[feat].to_numpy(dtype=float)
    print(f"{os.path.basename(path)}: {df.shape}, NaN {int(np.isnan(X).sum())}, inf {int(np.isinf(X).sum())}")
    assert (df[[c for c in feat if not c.startswith("g6_")]] >= 0).all().all(), "g1~g5에 음수가 있음 → log 변환 규칙 확인"
    return df

mon, unmon = load_features("mon", TRIM), load_features(UNMON_NAME, TRIM)
FEATS_ALL = [c for c in mon.columns if c[:3] in ("g1_", "g2_", "g3_", "g4_", "g5_", "g6_") and c not in DUP_DROP]
GROUPS = {g: [c for c in FEATS_ALL if c.startswith(g + "_")] for g in ["g1", "g2", "g3", "g4", "g5", "g6"]}
LOG_COLS = set(c for c in FEATS_ALL if not c.startswith("g6_"))
print("SVM 전체 특징:", len(FEATS_ALL), {g: len(v) for g, v in GROUPS.items()})

skew = mon[FEATS_ALL].skew().sort_values(ascending=False)
print("왜도 상위 5개 (log 변환 전):"); print(skew.head().round(1).to_string())
const_mon = [c for c in FEATS_ALL if mon[c].nunique() == 1]
print("mon trim0에서 상수인 특징:", const_mon, "→ 파이프라인의 VarianceThreshold가 학습 구간 기준으로 제거")

features_mon_trim0.csv: (19000, 116), NaN 0, inf 0
features_unmon10k_trim0.csv: (10000, 116), NaN 0, inf 0
SVM 전체 특징: 109 {'g1': 3, 'g2': 17, 'g3': 6, 'g4': 3, 'g5': 30, 'g6': 50}
왜도 상위 5개 (log 변환 전):
g5_iat_all_q50    137.8
g5_iat_in_q50     101.3
g5_iat_out_q25     22.3
g5_iat_out_q75     18.5
g5_iat_in_q75      17.2
mon trim0에서 상수인 특징: ['g5_iat_all_q25', 'g5_iat_in_q25', 'g6_cumul_00'] → 파이프라인의 VarianceThreshold가 학습 구간 기준으로 제거


## 2. 학습 / 최종 평가 분할 (모델팀 공유)

LR 노트북이 만든 `split_seed42.csv`를 그대로 읽는다. `record_id` 기준이라 trim0/10/20 모두 같은 분할이 적용된다.
최종 평가 기록은 특징 구성·C·gamma 선택에 쓰지 않는다.

In [5]:
split = pd.read_csv(SPLIT_PATH)
SPLIT = dict(zip(split.record_id, split.split))

def attach_split(df):
    s = df.record_id.map(SPLIT)
    assert s.notna().all(), "분할 파일에 없는 record_id가 있음"
    df = df.copy(); df["split"] = s.to_numpy()
    return df

mon, unmon = attach_split(mon), attach_split(unmon)
_all = pd.concat([mon, unmon], ignore_index=True)
print(pd.crosstab(_all.dataset, _all.split))

split     test  train
dataset              
mon       3800  15200
unmon10k  2000   8000


## 3. 공통 함수

**파이프라인** (5-fold의 각 학습 구간에서만 적합해 정보 유출을 막는다)
1. g1~g5 특징에 `log1p` 적용 (g6 누적값은 그대로)
2. `VarianceThreshold`: 학습 구간에서 상수인 특징 제거 (trim·데이터마다 상수 특징이 다르다)
3. `StandardScaler`
4. `SVC(kernel="rbf")`, 다중 분류는 one-vs-one

**C·gamma 탐색** (`tune`)
- 1차: log2 단위 2칸 간격 격자를 5-fold로 평가한다.
- 최적값이 탐색 범위의 끝이면 그 방향으로 2칸씩 넓혀 다시 평가한다(최대 2번).
- 2차: 최적값 주변을 1칸 간격으로 세밀하게 평가한다.
- `fixed=(log2C, log2gamma)`를 주면 탐색 없이 그 설정의 5-fold 점수만 구한다(같은 설정으로 조건을 비교하는 7·9번 실험).

Open-world 이진의 ROC·PR 곡선은 `decision_function` 점수로 계산한다. 확률 추정(`probability=True`)은 학습 시간이 크게 늘고 이진 곡선에는 필요하지 않아 쓰지 않는다. 판정 기준은 LR(확률 0.5)과 맞춰 SVM 기본 기준(점수 0)을 쓴다.

In [6]:
def safe_log1p(X):
    return np.log1p(np.clip(X, 0, None))

def make_svm(feats, C=1.0, gamma="scale", class_weight=None):
    log_cols = [c for c in feats if c in LOG_COLS]
    pre = ColumnTransformer([("log", FunctionTransformer(safe_log1p, feature_names_out="one-to-one"), log_cols)],
                            remainder="passthrough", verbose_feature_names_out=False)
    return Pipeline([("pre", pre), ("var", VarianceThreshold()), ("sc", StandardScaler()),
                     ("svc", SVC(kernel="rbf", C=C, gamma=gamma, class_weight=class_weight, cache_size=1000))])

CV = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
CV_TABLES = {}

def _grid_eval(X, y, pairs, class_weight):
    grid = [{"svc__C": [2.0 ** c], "svc__gamma": [2.0 ** g]} for c, g in pairs]
    gs = GridSearchCV(make_svm(list(X.columns), class_weight=class_weight), grid, cv=CV,
                      scoring="accuracy", n_jobs=N_JOBS, refit=False)
    gs.fit(X, y)
    out = {}
    for p, m, s, ft in zip(gs.cv_results_["params"], gs.cv_results_["mean_test_score"],
                           gs.cv_results_["std_test_score"], gs.cv_results_["mean_fit_time"]):
        out[(int(round(np.log2(p["svc__C"]))), int(round(np.log2(p["svc__gamma"]))))] = (m, s, ft)
    return out

def tune(X, y, name, log2C=None, log2G=None, class_weight=None, fixed=None, max_extend=2):
    t0 = time.time()
    tried = {}
    def run(pairs):
        new = [p for p in pairs if p not in tried]
        if new:
            tried.update(_grid_eval(X, y, new, class_weight))
    best = lambda: max(tried, key=lambda k: tried[k][0])
    if fixed is not None:
        run([tuple(fixed)])
    else:
        log2C = LOG2_C_COARSE if log2C is None else log2C
        log2G = LOG2_G_COARSE if log2G is None else log2G
        run(list(itertools.product(log2C, log2G)))
        for _ in range(max_extend):                       # 경계에 걸리면 범위 확장
            c, g = best()
            cs = sorted({k[0] for k in tried}); gs_ = sorted({k[1] for k in tried})
            ext = []
            if c == cs[0]:  ext += [(c - 2, gg) for gg in gs_]
            if c == cs[-1]: ext += [(c + 2, gg) for gg in gs_]
            if g == gs_[0]:  ext += [(cc, g - 2) for cc in cs]
            if g == gs_[-1]: ext += [(cc, g + 2) for cc in cs]
            if not ext: break
            print(f"  [{name}] 최적값 (log2C={c}, log2γ={g})이 경계 → 범위 확장")
            run(ext)
        c, g = best()
        run([(c + dc, g + dg) for dc in (-1, 0, 1) for dg in (-1, 0, 1)])   # 1칸 간격 세밀 탐색
    t_search = time.time() - t0
    c, g = best()
    tab = pd.DataFrame([dict(log2C=k[0], log2gamma=k[1], cv_acc_mean=v[0], cv_acc_std=v[1], fit_time_per_fold=v[2])
                        for k, v in tried.items()]).sort_values("cv_acc_mean", ascending=False).reset_index(drop=True)
    CV_TABLES[name] = tab.assign(experiment=name)
    info = dict(log2C=c, log2gamma=g, best_C=2.0 ** c, best_gamma=2.0 ** g,
                cv_acc=tried[(c, g)][0], cv_std=tried[(c, g)][1], n_evaluated=len(tried), search_time_s=t_search)
    print(f"[{name}] best log2C={c}, log2γ={g} | CV acc = {info['cv_acc']:.4f} ± {info['cv_std']:.4f} "
          f"| 평가한 조합 {len(tried)}개 | 탐색 {t_search:.0f}s")
    if fixed is None:
        print(tab.head(8).round(4).to_string())
    return info

def refit(X, y, info, class_weight=None):
    model = make_svm(list(X.columns), C=info["best_C"], gamma=info["best_gamma"], class_weight=class_weight)
    t0 = time.time(); model.fit(X, y); info["refit_time_s"] = time.time() - t0
    info["n_support"] = int(model[-1].n_support_.sum())
    return model

def timed_predict(model, X):
    t0 = time.time(); p = model.predict(X); return p, time.time() - t0

RESULTS, PREDS = [], {}

## 4. 공통 설정 (실험 3·4·5)

- `FEATS_110`: 다른 모델과 같은 110개. 중복 열 `g5_iat_pos_max`도 g1~g5 규칙대로 `log1p`를 적용한다.
- `FEATS_A`: 개선 특징 후보(g1 + g6, 53개). 실험 3에서 공통 기준과 구분해 기록한다.

In [7]:
FEATS_110 = [c for c in mon.columns if c[:3] in ("g1_", "g2_", "g3_", "g4_", "g5_", "g6_")]
assert len(FEATS_110) == 110
LOG_COLS.update(c for c in FEATS_110 if not c.startswith("g6_"))
FEATS_A = GROUPS["g1"] + GROUPS["g6"]
print("공통 기준:", len(FEATS_110), "개 / 개선 후보 A:", len(FEATS_A), "개")

def binary_view(mon, unmon):
    df = pd.concat([mon, unmon], ignore_index=True)
    df["y"] = np.where(df.dataset == "mon", 1, -1)
    return df

def openworld_metrics(y_true_bin, y_pred_bin):
    tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin, labels=[-1, 1]).ravel()
    return dict(mon_as_mon=tp / (tp + fn), unmon_as_unmon=tn / (tn + fp),
                precision=tp / (tp + fp) if tp + fp else 0.0, fpr=fp / (fp + tn), TP=tp, FN=fn, TN=tn, FP=fp)

import matplotlib
if not IN_COLAB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
def show_or_close():
    if IN_COLAB: plt.show()
    else: plt.close()

공통 기준: 110 개 / 개선 후보 A: 53 개


## 5. 실험 3: 하이퍼파라미터 적정성 (Closed-world)

**확인할 것**
1. C에 따라 정확도가 어떻게 변하는가 (gamma 고정): 공통 기준 110개(γ = 2⁻⁶)와 개선 후보 A(γ = 2⁻²) 각각
2. gamma에 따라 어떻게 변하는가 (110개, C = 2⁶ 고정)
3. 기본 설정(`C=1, gamma="scale"`)은 어디쯤인가

대규모 재탐색은 하지 않고, 기존 최적값을 지나가는 한 줄짜리 곡선만 그린다. 모든 점은 같은 분할의 5-fold CV 정확도다.

In [8]:
def cv_point(X, y, C, gamma):
    t0 = time.time()
    r = cross_validate(make_svm(list(X.columns), C=C, gamma=gamma), X, y, cv=CV, scoring="accuracy", n_jobs=N_JOBS)
    return r["test_score"].mean(), r["test_score"].std(), (time.time() - t0) / N_FOLDS

tr = mon[mon.split == "train"]
rows = []
for name, feats, lg in [("공통 110개", FEATS_110, -6), ("개선 A (g1+g6)", FEATS_A, -2)]:
    m, s, t = cv_point(tr[feats], tr.label, 1.0, "scale")
    rows.append(dict(sweep="default", features=name, log2C=0, log2gamma="scale", cv_acc=m, cv_std=s, sec_per_fold=t))
    print(f"[{name}] 기본 설정 C=1, gamma=scale: {m:.4f} ± {s:.4f}")
    for lc in [0, 2, 4, 6, 8, 10, 12, 14]:
        m, s, t = cv_point(tr[feats], tr.label, 2.0 ** lc, 2.0 ** lg)
        rows.append(dict(sweep="C", features=name, log2C=lc, log2gamma=lg, cv_acc=m, cv_std=s, sec_per_fold=t))
        print(f"[{name}] log2C={lc:>2}, log2γ={lg}: {m:.4f} ± {s:.4f} ({t:.1f}s/fold)")
for lg in [-10, -8, -6, -4, -2]:
    m, s, t = cv_point(tr[FEATS_110], tr.label, 2.0 ** 6, 2.0 ** lg)
    rows.append(dict(sweep="gamma", features="공통 110개", log2C=6, log2gamma=lg, cv_acc=m, cv_std=s, sec_per_fold=t))
    print(f"[공통 110개] log2C=6, log2γ={lg:>3}: {m:.4f} ± {s:.4f} ({t:.1f}s/fold)")
exp3 = pd.DataFrame(rows)
exp3.to_csv(os.path.join(RESULT_DIR, "svm_exp3_sensitivity.csv"), index=False)

# 표준화 후 gamma="scale"의 실제 값: 1 / (유효 특징 수 × 분산 1)
for name, feats in [("공통 110개", FEATS_110), ("개선 A", FEATS_A)]:
    n_eff = int((tr[feats].nunique() > 1).sum())
    print(f"{name}: gamma='scale' ≈ 1/{n_eff} = 2^{-np.log2(n_eff):.2f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, en, mk in [("공통 110개", "common 110 feats", "o"), ("개선 A (g1+g6)", "improved A: g1+g6 (53)", "s")]:
    d = exp3[(exp3.sweep == "C") & (exp3.features == name)]
    ax[0].errorbar(d.log2C, d.cv_acc, yerr=d.cv_std, marker=mk, capsize=3, label=f"{en}, gamma=2^{int(d.log2gamma.iloc[0])}")
ax[0].set_xlabel("log2 C"); ax[0].set_ylabel("5-fold CV accuracy"); ax[0].set_title("C sensitivity (closed-world)"); ax[0].legend(); ax[0].grid(alpha=.3)
d = exp3[exp3.sweep == "gamma"]
ax[1].errorbar(d.log2gamma, d.cv_acc, yerr=d.cv_std, marker="o", capsize=3)
ax[1].set_xlabel("log2 gamma"); ax[1].set_title("gamma sensitivity (110 features, C=2^6)"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(RESULT_DIR, "svm_exp3_sensitivity.png"), dpi=150); show_or_close()
print("그림 저장: svm_exp3_sensitivity.png")

[공통 110개] 기본 설정 C=1, gamma=scale: 0.6394 ± 0.0020
[공통 110개] log2C= 0, log2γ=-6: 0.6949 ± 0.0023 (6.1s/fold)
[공통 110개] log2C= 2, log2γ=-6: 0.8035 ± 0.0047 (5.8s/fold)
[공통 110개] log2C= 4, log2γ=-6: 0.8402 ± 0.0068 (5.7s/fold)
[공통 110개] log2C= 6, log2γ=-6: 0.8420 ± 0.0052 (5.8s/fold)
[공통 110개] log2C= 8, log2γ=-6: 0.8422 ± 0.0052 (5.8s/fold)
[공통 110개] log2C=10, log2γ=-6: 0.8422 ± 0.0052 (5.7s/fold)
[공통 110개] log2C=12, log2γ=-6: 0.8422 ± 0.0052 (5.9s/fold)
[공통 110개] log2C=14, log2γ=-6: 0.8422 ± 0.0052 (5.8s/fold)
[개선 A (g1+g6)] 기본 설정 C=1, gamma=scale: 0.2955 ± 0.0053
[개선 A (g1+g6)] log2C= 0, log2γ=-2: 0.5538 ± 0.0067 (5.1s/fold)
[개선 A (g1+g6)] log2C= 2, log2γ=-2: 0.6944 ± 0.0070 (4.6s/fold)
[개선 A (g1+g6)] log2C= 4, log2γ=-2: 0.7878 ± 0.0102 (4.4s/fold)
[개선 A (g1+g6)] log2C= 6, log2γ=-2: 0.8341 ± 0.0071 (4.4s/fold)
[개선 A (g1+g6)] log2C= 8, log2γ=-2: 0.8486 ± 0.0080 (4.4s/fold)
[개선 A (g1+g6)] log2C=10, log2γ=-2: 0.8511 ± 0.0065 (4.4s/fold)
[개선 A (g1+g6)] log2C=12, log2γ=-2: 0.8521 ± 0.0062 (4

## 6. 실험 4: 동일 조건 비교 (trim0, 특징 110개)

과제마다 5-fold CV로 C·gamma를 고른 뒤 학습 데이터 전체로 재학습하고 test로 한 번 평가한다. 탐색은 기존 109개 실험의 최적값 주변으로 시작한다(경계면 1회 확장 후 1칸 간격 세밀 탐색).

| 과제 | 탐색 시작 범위 (log2C × log2γ) |
| --- | --- |
| Closed-world | {4, 6, 8} × {−7, −6, −5} |
| Open-world 이진 | {1, 3, 5} × {−5, −4, −3} |
| Open-world 다중 | Closed 최적값 기준 {C−2, C, C+2} × {γ−1, γ, γ+1} |

In [9]:
EXP4 = []
F = FEATS_110

# Closed-world
tr, te = mon[mon.split == "train"], mon[mon.split == "test"]
info = tune(tr[F], tr.label, "exp4_closed", log2C=[4, 6, 8], log2G=[-7, -6, -5], max_extend=1)
model = refit(tr[F], tr.label, info)
pred, t_pred = timed_predict(model, te[F])
EXP4.append(dict(task="closed_world", **info, test_acc=accuracy_score(te.label, pred),
                 test_macro_f1=f1_score(te.label, pred, average="macro"), predict_time_s=t_pred))
PREDS["exp4_closed"] = te[["record_id", "label"]].assign(pred=pred)
print(f"→ Closed-world test acc {EXP4[-1]['test_acc']:.4f} | macro-F1 {EXP4[-1]['test_macro_f1']:.4f}")
BEST4_CLOSED = (info["log2C"], info["log2gamma"])

  [exp4_closed] 최적값 (log2C=8, log2γ=-6)이 경계 → 범위 확장
[exp4_closed] best log2C=7, log2γ=-6 | CV acc = 0.8422 ± 0.0051 | 평가한 조합 18개 | 탐색 455s
   log2C  log2gamma  cv_acc_mean  cv_acc_std  fit_time_per_fold
0      7         -6       0.8422      0.0051             4.1664
1      8         -6       0.8422      0.0052             4.0641
2     10         -6       0.8422      0.0052             4.1300
3      9         -6       0.8422      0.0052             4.0993
4      6         -6       0.8420      0.0052             4.1340
5      7         -5       0.8414      0.0063             5.0113
6      9         -5       0.8414      0.0063             5.0694
7     10         -5       0.8414      0.0063             5.0276
→ Closed-world test acc 0.8521 | macro-F1 0.8525


In [10]:
# Open-world 이진
bv = binary_view(mon, unmon)
trb, teb = bv[bv.split == "train"], bv[bv.split == "test"]
info = tune(trb[F], trb.y, "exp4_binary", log2C=[1, 3, 5], log2G=[-5, -4, -3], max_extend=1)
model_b = refit(trb[F], trb.y, info)
pred, t_pred = timed_predict(model_b, teb[F])
score_te = model_b.decision_function(teb[F])
m = openworld_metrics(teb.y, pred)
EXP4.append(dict(task="open_binary", **info, test_acc=accuracy_score(teb.y, pred),
                 majority_baseline=(teb.y == 1).mean(), roc_auc=roc_auc_score(teb.y, score_te),
                 pr_auc=average_precision_score(teb.y, score_te, pos_label=1), predict_time_s=t_pred, **m))
PREDS["exp4_binary"] = teb[["record_id", "dataset", "y"]].assign(pred=pred, score=score_te)
BEST4_BINARY = (info["log2C"], info["log2gamma"])
r = EXP4[-1]
print(f"→ 이진 test acc {r['test_acc']:.4f} | mon→mon {r['mon_as_mon']:.4f} | unmon→unmon {r['unmon_as_unmon']:.4f} | "
      f"precision {r['precision']:.4f} | FPR {r['fpr']:.4f} | ROC-AUC {r['roc_auc']:.4f}")

/usr/local/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
[exp4_binary] best log2C=3, log2γ=-4 | CV acc = 0.8913 ± 0.0040 | 평가한 조합 15개 | 탐색 946s
   log2C  log2gamma  cv_acc_mean  cv_acc_std  fit_time_per_fold
0      3         -4       0.8913      0.0040            17.5457
1      2         -3       0.8904      0.0044            19.4328
2      4         -4       0.8897      0.0037            18.4437
3      5         -5       0.8894      0.0050            18.5015
4      4         -5       0.8890      0.0050            15.8784
5      3         -3       0.8887 

In [11]:
# Open-world 다중 (96 클래스)
dm = pd.concat([mon, unmon], ignore_index=True)
trm, tem = dm[dm.split == "train"], dm[dm.split == "test"]
c0, g0 = BEST4_CLOSED
info = tune(trm[F], trm.label, "exp4_multi", log2C=[c0 - 2, c0, c0 + 2], log2G=[g0 - 1, g0, g0 + 1], max_extend=1)
model_m = refit(trm[F], trm.label, info)
pred, t_pred = timed_predict(model_m, tem[F])
y = tem.label.to_numpy(); is_mon = y >= 0
m = openworld_metrics(np.where(y >= 0, 1, -1), np.where(pred >= 0, 1, -1))
EXP4.append(dict(task="open_multi", **info, test_acc=accuracy_score(y, pred), mon_site_exact=(pred[is_mon] == y[is_mon]).mean(),
                 test_macro_f1=f1_score(y, pred, average="macro"), predict_time_s=t_pred, **m))
PREDS["exp4_multi"] = tem[["record_id", "dataset", "label"]].assign(pred=pred)
r = EXP4[-1]
print(f"→ 다중 test acc {r['test_acc']:.4f} | mon→mon {r['mon_as_mon']:.4f} | unmon→unmon {r['unmon_as_unmon']:.4f} | "
      f"mon 사이트 정답 {r['mon_site_exact']:.4f}")

[exp4_multi] best log2C=7, log2γ=-6 | CV acc = 0.8294 ± 0.0040 | 평가한 조합 15개 | 탐색 1150s
   log2C  log2gamma  cv_acc_mean  cv_acc_std  fit_time_per_fold
0      7         -6       0.8294      0.0040            17.7331
1      6         -6       0.8281      0.0038            17.0946
2      8         -6       0.8281      0.0043            16.9713
3      6         -5       0.8281      0.0021            18.2395
4      7         -5       0.8278      0.0025            18.7755
5      9         -6       0.8277      0.0047            17.5783
6      8         -5       0.8276      0.0027            18.2777
7      9         -5       0.8276      0.0027            18.4956
→ 다중 test acc 0.8384 | mon→mon 0.8905 | unmon→unmon 0.9005 | mon 사이트 정답 0.8058


## 7. 실험 5: Open-world 이진 임계값 분석

**절차** (최종 평가 데이터는 임계값 선택에 쓰지 않음)
1. 실험 4 이진 설정으로, 학습 데이터의 5-fold **교차 예측 점수**(`decision_function`, 각 기록은 자신이 빠진 fold의 모델로 예측)를 만든다.
2. 목표 FPR(1%·5%·10%)마다, 교차 예측에서 unmon 점수의 상위 FPR 지점을 임계값으로 정한다. (점수 > 임계값이면 mon)
3. 실험 4에서 학습 데이터 전체로 재학습한 모델의 test 점수에 그 임계값을 **한 번만** 적용한다.
4. 기본 기준(점수 0)도 함께 기록한다.

목표 FPR은 팀 합의 전이므로 세 값을 모두 기록한다. 다른 모델도 같은 절차(학습 데이터 5-fold 교차 예측 점수로 임계값 결정)를 쓰면 같은 기준으로 비교할 수 있다.

In [12]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_curve, precision_recall_curve

t0 = time.time()
oof = cross_val_predict(make_svm(F, C=2.0 ** BEST4_BINARY[0], gamma=2.0 ** BEST4_BINARY[1]), trb[F], trb.y,
                        cv=CV, method="decision_function", n_jobs=N_JOBS)
print(f"교차 예측 점수 계산 {time.time() - t0:.0f}s")
ytr = trb.y.to_numpy(); yte = teb.y.to_numpy()

def at_threshold(y, s, thr):
    p = np.where(s > thr, 1, -1)
    return dict(acc=accuracy_score(y, p), **openworld_metrics(y, p))

rows = []
for target in [None, 0.01, 0.05, 0.10]:
    thr = 0.0 if target is None else float(np.quantile(oof[ytr == -1], 1 - target))
    v, t = at_threshold(ytr, oof, thr), at_threshold(yte, score_te, thr)
    rows.append(dict(target_fpr="기본(점수 0)" if target is None else f"{target:.0%}", threshold=thr,
                     val_tpr=v["mon_as_mon"], val_fpr=v["fpr"], val_precision=v["precision"],
                     test_tpr=t["mon_as_mon"], test_fpr=t["fpr"], test_precision=t["precision"], test_acc=t["acc"],
                     TP=t["TP"], FN=t["FN"], TN=t["TN"], FP=t["FP"]))
exp5 = pd.DataFrame(rows)
print(exp5.round(4).to_string(index=False))
exp5.to_csv(os.path.join(RESULT_DIR, "svm_exp5_threshold.csv"), index=False)
pd.DataFrame({"record_id": trb.record_id.values, "y": ytr, "oof_score": oof}).to_csv(
    os.path.join(RESULT_DIR, "svm_exp5_oof_scores.csv"), index=False)
print(f"교차 예측 ROC-AUC {roc_auc_score(ytr, oof):.4f} | test ROC-AUC {roc_auc_score(yte, score_te):.4f}")

fpr_c, tpr_c, _ = roc_curve(yte, score_te)
prec_c, rec_c, _ = precision_recall_curve(yte, score_te, pos_label=1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(fpr_c, tpr_c, label=f"test ROC (AUC {roc_auc_score(yte, score_te):.3f})")
ax[1].plot(rec_c, prec_c, label=f"test PR (AP {average_precision_score(yte, score_te, pos_label=1):.3f})")
for _, r in exp5.iterrows():
    lab = "default (score 0)" if r.target_fpr.startswith("기본") else f"target FPR {r.target_fpr}"
    ax[0].scatter(r.test_fpr, r.test_tpr, zorder=3, label=f"{lab}: TPR {r.test_tpr:.2f}, FPR {r.test_fpr:.2f}")
    ax[1].scatter(r.test_tpr, r.test_precision, zorder=3)
ax[0].plot([0, 1], [0, 1], ls=":", c="gray")
ax[0].set_xlabel("FPR (unmon → mon)"); ax[0].set_ylabel("TPR (mon → mon)"); ax[0].set_title("SVM open-world binary: ROC"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
ax[1].set_xlabel("Recall (TPR)"); ax[1].set_ylabel("Precision"); ax[1].set_title("Precision-Recall"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(RESULT_DIR, "svm_exp5_roc_pr.png"), dpi=150); show_or_close()
print("그림 저장: svm_exp5_roc_pr.png")

교차 예측 점수 계산 63s
target_fpr  threshold  val_tpr  val_fpr  val_precision  test_tpr  test_fpr  test_precision  test_acc   TP   FN   TN  FP
  기본(점수 0)     0.0000   0.9359   0.1935         0.9019    0.9416    0.1835          0.9070    0.8984 3578  222 1633 367
        1%     1.5036   0.3566   0.0100         0.9855    0.3671    0.0080          0.9887    0.5826 1395 2405 1984  16
        5%     0.8276   0.7322   0.0500         0.9653    0.7382    0.0505          0.9652    0.8110 2805  995 1899 101
       10%     0.4404   0.8599   0.1000         0.9423    0.8647    0.0995          0.9429    0.8771 3286  514 1801 199
교차 예측 ROC-AUC 0.9487 | test ROC-AUC 0.9524
그림 저장: svm_exp5_roc_pr.png


## 8. 저장

In [13]:
exp4 = pd.DataFrame(EXP4)
cols = ["task", "log2C", "log2gamma", "cv_acc", "cv_std", "test_acc", "majority_baseline", "mon_as_mon", "unmon_as_unmon",
        "mon_site_exact", "precision", "fpr", "roc_auc", "pr_auc", "test_macro_f1", "TP", "FN", "TN", "FP",
        "n_support", "search_time_s", "refit_time_s", "predict_time_s"]
exp4 = exp4[[c for c in cols if c in exp4.columns]]
print(exp4.round(4).to_string(index=False))
exp4.to_csv(os.path.join(RESULT_DIR, "svm_exp4_summary.csv"), index=False)
pd.concat([CV_TABLES[k] for k in CV_TABLES if k.startswith("exp4")], ignore_index=True).to_csv(
    os.path.join(RESULT_DIR, "svm_exp4_cv_tables.csv"), index=False)
for k in ["exp4_closed", "exp4_binary", "exp4_multi"]:
    PREDS[k].to_csv(os.path.join(RESULT_DIR, f"svm_pred_{k}.csv"), index=False)
print("저장 완료")

        task  log2C  log2gamma  cv_acc  cv_std  test_acc  majority_baseline  mon_as_mon  unmon_as_unmon  mon_site_exact  precision    fpr  roc_auc  pr_auc  test_macro_f1     TP    FN     TN    FP  n_support  search_time_s  refit_time_s  predict_time_s
closed_world      7         -6  0.8422  0.0051    0.8521                NaN         NaN             NaN             NaN        NaN    NaN      NaN     NaN         0.8525    NaN   NaN    NaN   NaN      12245       455.0683        5.6191          8.3431
 open_binary      3         -4  0.8913  0.0040    0.8984             0.6552      0.9416          0.8165             NaN     0.9070 0.1835   0.9524  0.9706            NaN 3578.0 222.0 1633.0 367.0       9235       945.6354       28.9421          6.1971
  open_multi      7         -6  0.8294  0.0040    0.8384                NaN      0.8905          0.9005          0.8058     0.9445 0.0995      NaN     NaN         0.8286 3384.0 416.0 1801.0 199.0      17289      1150.3693       24.5047         